In [5]:
import pandas as pd
import numpy as np
print("imported numpy and pandas")

imported numpy and pandas


In [6]:
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, classification_report
print("loaded libraries")

loaded libraries


In [7]:
# initialize hardware
import os
import torch
os.environ['CUDA_LAUNCH_BLOCKING'] = "1" # Force synchronous errors for debugging
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    print(f"Data source import complete. Running on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Data source import complete. Running on CPU (GPU was not detected by PyTorch).")

torch.cuda.empty_cache()
model_name = 'facebook/bart-large-mnli'
sbert_encoder = SentenceTransformer(model_name)

Data source import complete. Running on GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

[transformers] BartModel LOAD REPORT from: facebook/bart-large-mnli
Key                                 | Status     |  | 
------------------------------------+------------+--+-
classification_head.dense.weight    | UNEXPECTED |  | 
classification_head.out_proj.weight | UNEXPECTED |  | 
classification_head.dense.bias      | UNEXPECTED |  | 
classification_head.out_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
import kagglehub

# Download and Load Data
path = kagglehub.dataset_download('timospinde/babe-media-bias-annotations-by-experts')

# load data
base = "C:/Users/robin/.cache/kagglehub/datasets/timospinde/babe-media-bias-annotations-by-experts/versions/1/data"
neutral_file = base + "/news_headlines_usa_neutral.csv"
neutral_df = pd.read_csv(neutral_file)

print(neutral_df.head())

biased_file = base + "/news_headlines_usa_biased.csv"
biased_df = pd.read_csv(biased_file)

print(biased_df.head())

   stories_id         publish_date  \
0   663692044  2020-07-12 08:00:00   
1   668844035  2020-07-11 08:00:00   
2   669761491  2020-07-10 08:00:00   
3   680460617  2020-11-05 07:00:00   
4   803694205  2020-06-15 00:00:00   

                                               title  \
0  "Don't Tase Me, Bro" tops '07 memorable quote ...   
1  U.S. healthcare falls short in survey of 7 nat...   
2  Nicotine may ease Parkinson's symptoms: U.S. s...   
3  Experts Divided Over Safety of Indian Point Nu...   
4  Reports: Petraeus off the list, Trump down to ...   

                                                 url language  ap_syndicated  \
0  http://www.reuters.com/article/us-quotes-odd-i...       en          False   
1  http://www.reuters.com/article/us-healthcare-s...       en          False   
2  https://www.reuters.com/article/us-parkinsons-...       en          False   
3  http://www.reuters.com/article/idUS38169798202...       en          False   
4  http://thehill.com/blogs/blog-b

In [9]:
from sklearn.model_selection import train_test_split

# Ensure result labels are assigned
neutral_df['result'] = 'neutral'
biased_df['result'] = 'biased'

full_df = pd.concat([neutral_df, biased_df], ignore_index=True)
full_df = full_df.drop_duplicates(subset=['title'])

train_df, temp_df = train_test_split(
    full_df,
    test_size=0.2,
    random_state=2347,
    stratify=full_df['result']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=2347,
    stratify=temp_df['result']
)

# shuffle for training
train_df_shuffled = train_df.sample(frac=1, random_state=2347).reset_index(drop=True)
val_df_shuffled = val_df.sample(frac=1, random_state=2347).reset_index(drop=True)
test_df_shuffled = test_df.sample(frac=1, random_state=2347).reset_index(drop=True)

print("Data Splits (Deduplicated & Balanced):")
print(f"Training shape: {train_df_shuffled.shape}")
print(f"Validation shape: {val_df_shuffled.shape}")
print(f"Testing shape:  {test_df_shuffled.shape}")

# verify no overlaps
overlap = set(train_df_shuffled['title']).intersection(set(val_df_shuffled['title']))
print(f"\nLeakage check (Overlapping titles): {len(overlap)}")

Data Splits (Deduplicated & Balanced):
Training shape: (96846, 11)
Validation shape: (12106, 11)
Testing shape:  (12106, 11)

Leakage check (Overlapping titles): 0


In [10]:
# Split the shuffled training data into actual training and validation sets
train_df_nn, val_df_nn = train_test_split(train_df_shuffled, test_size=0.15, random_state=42, stratify=train_df_shuffled['result'])

print(f"Training data for NN shape: {train_df_nn.shape}")
print(f"Validation data for NN shape: {val_df_nn.shape}")

Training data for NN shape: (82319, 11)
Validation data for NN shape: (14527, 11)


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

# This function uses S-BERT embeddings and global centroids
def calculate_bias_memberships_sbert(sbert_features):
    # Ensure centroids are accessible (assuming they are global or passed)
    if 'global_biased_centroid' not in globals() or 'global_neutral_centroid' not in globals():
        raise ValueError("Centroids 'global_biased_centroid' and 'global_neutral_centroid' are not defined globally.")

    sbert_features_reshaped = sbert_features.reshape(1, -1)

    sim_to_biased = cosine_similarity(sbert_features_reshaped, global_biased_centroid.reshape(1, -1))[0][0]
    sim_to_neutral = cosine_similarity(sbert_features_reshaped, global_neutral_centroid.reshape(1, -1))[0][0]

    # Scale similarities from [-1, 1] to [0, 1]
    similarity_biased = (sim_to_biased + 1) / 2
    similarity_neutral = (sim_to_neutral + 1) / 2

    # Determine a net bias score
    net_bias_score = similarity_biased - similarity_neutral
    # Scale net_bias_score from [-1, 1] to [0, 1] for easier interpretation
    scaled_net_bias_score = (net_bias_score + 1) / 2


    # Calculate certainty and indeterminacy based on distance from 0.5 (perfect ambiguity)
    certainty = abs(scaled_net_bias_score - 0.5) * 2  # 0 when score is 0.5, 1 when score is 0 or 1
    indeterminacy = 1 - certainty # 1 when score is 0.5, 0 when score is 0 or 1

    # Adjust T, F values to be more sensitive to certainty
    # When certainty is low (indeterminacy is high), T and F approach 0.5
    # When certainty is high (indeterminacy is low), T and F approach scaled_net_bias_score or 1-scaled_net_bias_score

    # For 'biased' membership:
    T_b = scaled_net_bias_score * certainty + (0.5 * indeterminacy)
    F_b = (1 - scaled_net_bias_score) * certainty + (0.5 * indeterminacy)
    I_b = indeterminacy

    # For 'neutral' membership:
    T_n = (1 - scaled_net_bias_score) * certainty + (0.5 * indeterminacy)
    F_n = scaled_net_bias_score * certainty + (0.5 * indeterminacy)
    I_n = indeterminacy

    return {
        "biased": (T_b, I_b, F_b),
        "neutral": (T_n, I_n, F_n)
    }

def classify_article(memberships):
    # Neutrosophic Score: T - I - F
    scores = {label: (T - I - F) for label, (T, I, F) in memberships.items()}
    return max(scores, key=scores.get)

print("Cell Complete")

Cell Complete


In [12]:
print("Generating S-BERT embeddings for NN validation data...")
val_embeddings_nn = sbert_encoder.encode(val_df_nn['title'].tolist(), show_progress_bar=True, convert_to_numpy=True)

print("Generating S-BERT embeddings for training data...")
train_embeddings = sbert_encoder.encode(train_df_shuffled['title'].tolist(), show_progress_bar=True, convert_to_numpy=True)

Generating S-BERT embeddings for NN validation data...


Batches:   0%|          | 0/454 [00:00<?, ?it/s]

Generating S-BERT embeddings for training data...


Batches:   0%|          | 0/3027 [00:00<?, ?it/s]

In [13]:
import torch
import torch.nn.functional as F


# custom nueral network
class ContrastiveLoss(torch.nn.Module):
    def __init__(self, margin=2.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        # output1, output2 are embeddings (from Sentence-BERT)
        # label =  0 for similar, 1 for dissimilar

        # map the inputs into the euclidian space, where values outside the margin, defined in the initialization, are considered to be 1 and not similar, and 0 for all the labels inside margin
        euclidean_distance = F.pairwise_distance(output1, output2, keepdim=True)

        # the calculated loss per batch of embeddings
        loss_contrastive = torch.mean((1 - label) * torch.pow(euclidean_distance, 2) +
                                      (label) * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2))

        return loss_contrastive

print("class defined.")

class defined.


In [14]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random

# Define a Simple Model with dynamic dropout tuning support
class BiasedNeutralModel(nn.Module):
    def __init__(self, embedding_dim=384, num_classes=2, dropout_rate=0.3):
        super(BiasedNeutralModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(embedding_dim, 1024), # Expanded width
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, embedding_dim)
        )
        self.classifier = nn.Linear(embedding_dim, num_classes) # Classification head

    def forward(self, x):
        # x are the input S-BERT embeddings
        embeddings = self.network(x) # Pass through the network to get enhanced embeddings
        logits = self.classifier(embeddings)
        return logits, embeddings

# Custom Dataset for Classification and Contrastive Learning
class CombinedDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = embeddings
        self.labels = labels

        # Pre-compute indices for each label
        self.label_to_indices = {label.item(): (self.labels == label).nonzero(as_tuple=True)[0].tolist() for label in torch.unique(self.labels)}

        # Pre-compute all negative indices for each label to avoid repeated computation in __getitem__
        self.all_negative_indices = {}
        for current_label in torch.unique(self.labels):
            # Indices where the label is NOT current_label
            self.all_negative_indices[current_label.item()] = (self.labels != current_label).nonzero(as_tuple=True)[0].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        anchor_embedding = self.embeddings[idx]
        anchor_label_tensor = self.labels[idx]
        anchor_label = anchor_label_tensor.item() # Ensure anchor_label is an int for dictionary lookup

        # Get candidates for positive pairs (same label, but not the anchor itself)
        # Filter out the anchor's own index from the precomputed list
        potential_positives = [i for i in self.label_to_indices[anchor_label] if i != idx]

        # Get candidates for negative pairs using the precomputed list
        potential_negatives = self.all_negative_indices[anchor_label]

        pair_embedding = None
        pair_label = None

        # Prioritize generating a positive or negative pair as requested
        if random.random() > 0.5:  # Try to create a positive pair
            if len(potential_positives) > 0:
                pair_idx = random.choice(potential_positives)
                pair_label = torch.tensor(0.0) # Similar
            elif len(potential_negatives) > 0: # Fallback to negative if no other positive exists
                pair_idx = random.choice(potential_negatives)
                pair_label = torch.tensor(1.0) # not similar
            else: # very rare
                pair_idx = idx # pair with self
                pair_label = torch.tensor(0.0) # Treat as similar to self
        else:  # Try to create a negative pair
            if len(potential_negatives) > 0:
                pair_idx = random.choice(potential_negatives)
                pair_label = torch.tensor(1.0) # not similar
            elif len(potential_positives) > 0: # Fallback to positive if no negative exists
                pair_idx = random.choice(potential_positives)
                pair_label = torch.tensor(0.0)
            else:
                pair_idx = idx
                pair_label = torch.tensor(0.0)

        paired_embedding = self.embeddings[pair_idx]

        return anchor_embedding, anchor_label_tensor, paired_embedding, pair_label

print("Model and CombinedDataset classes defined with optimized __getitem__ and tunable dropout.")

Model and CombinedDataset classes defined with optimized __getitem__ and tunable dropout.


In [15]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from collections import Counter
import numpy as np

# 1. HARDWARE & RESET
torch.cuda.empty_cache() # Clear stale GPU memory
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. VALIDATE & ALIGN DATA
# Ensure labels are flat, within [0, 1] range, and match embedding length
label_map = {'biased': 0, 'neutral': 1}
train_labels_numeric = train_df_shuffled['result'].map(label_map).values.flatten()
N = min(len(train_labels_numeric), train_embeddings.shape[0])

# Strict validation
assert train_labels_numeric[:N].max() <= 1, "Labels contain values > 1!"
assert train_labels_numeric[:N].min() >= 0, "Labels contain negative values!"

# Load tensors
train_embeddings_tensor = torch.from_numpy(train_embeddings[:N]).float().to(device)
train_labels_tensor = torch.from_numpy(train_labels_numeric[:N]).long().to(device)

train_labels_tensor = train_labels_tensor.long()

# 3. INITIALIZE MODEL & WEIGHTS
embedding_dim = train_embeddings_tensor.shape[1]
num_classes = 2
combined_model = BiasedNeutralModel(embedding_dim=embedding_dim, num_classes=num_classes)
combined_model.to(device)

# Class weighting for imbalance
class_counts = Counter(train_labels_numeric[:N])
class_weights = torch.tensor(
    [len(train_labels_numeric[:N]) / (num_classes * class_counts[i]) for i in range(num_classes)],
    dtype=torch.float32
).to(device)

# 4. LOSSES & OPTIMIZERS
criterion_ce = nn.CrossEntropyLoss(weight=class_weights).to(device)
criterion_contrastive = ContrastiveLoss(margin=1.0).to(device)

c1 = nn.Parameter(torch.tensor(0.0, requires_grad=True, device=device))
c2 = nn.Parameter(torch.tensor(0.0, requires_grad=True, device=device))

optimizer_model = torch.optim.Adam(combined_model.parameters(), lr=0.0005)
optimizer_weights = torch.optim.Adam([c1, c2], lr=0.00001)

# 5. DATASET PREP (Using the safer DataLoader approach)
# This uses the CombinedDataset class you already have defined
train_dataset = CombinedDataset(train_embeddings_tensor, train_labels_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print("Block execution complete. Your model and data are aligned and ready.")

Using device: cuda
Block execution complete. Your model and data are aligned and ready.


In [16]:
# Prepare data for the validation loop

# Convert results to numerical labels for validation set
val_labels_numeric = val_df_nn['result'].map(label_map).values

# Convert embeddings and labels to PyTorch tensors for validation set
val_embeddings_tensor = torch.from_numpy(val_embeddings_nn).float()
val_labels_tensor = torch.from_numpy(val_labels_numeric).long()
val_labels_tensor = val_labels_tensor.long()


# Create the dataset and dataloader for validation
val_dataset = CombinedDataset(val_embeddings_tensor, val_labels_tensor)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False) # No need to shuffle validation data

print("Validation data prepared.")

Validation data prepared.


C:\Users\robin\AppData\Local\Temp\ipykernel_27156\1916127682.py:8: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  val_labels_tensor = torch.from_numpy(val_labels_numeric).long()


In [17]:
N = len(train_labels_tensor)
paired_embeddings_list = []
pair_labels_list = []

print("Pre-calculating pairs on GPU...")

for i in range(N):
    current_label = train_labels_tensor[i]
    # print("current label", current_label)

    # Vectorized match: find indices that share the same label
    matching_indices = (train_labels_tensor == current_label).nonzero(as_tuple=True)[0]

    # Pick a random matching index
    random_idx = matching_indices[torch.randint(0, len(matching_indices), (1,)).item()]

    paired_embeddings_list.append(train_embeddings_tensor[random_idx])
    pair_labels_list.append(train_labels_tensor[random_idx])

# Stack everything into the exact tensors the training loop is looking for
train_paired_embeddings = torch.stack(paired_embeddings_list).to(device)
train_pair_labels = torch.stack(pair_labels_list).to(device)

print("Tensors successfully created! You can now run your training loop.")

Pre-calculating pairs on GPU...
Tensors successfully created! You can now run your training loop.


In [18]:
import copy

optimized_model = combined_model

num_epochs = 20
best_val_accuracy = 0.0
best_model_wts = copy.deepcopy(optimized_model.state_dict())

num_train_samples = train_embeddings_tensor.size(0)
batch_size = 512

# move tensors to GPU
val_embeddings_tensor = val_embeddings_tensor.to(device)
val_labels_tensor = val_labels_tensor.to(device)

print(f"Training on {device}...")

for epoch in range(num_epochs):
    optimized_model.train()
    total_loss_epoch = 0
    total_ce_loss_epoch = 0
    total_contrastive_loss_epoch = 0

    # Create a randomized index order for shuffling entirely on GPU
    permutation = torch.randperm(num_train_samples, device=device)

    # Directly slice the GPU tensors in chunks
    for i in range(0, num_train_samples, batch_size):
        indices = permutation[i:i + batch_size]

        # Fast GPU memory slicing (takes virtually 0ms)
        anchor_embeds = train_embeddings_tensor[indices]
        anchor_labels = train_labels_tensor[indices]
        paired_embeds = train_paired_embeddings[indices]
        pair_labels = train_pair_labels[indices]

        # Zero gradients
        optimizer_model.zero_grad()
        optimizer_weights.zero_grad()

        # Forward pass
        logits_anchor, embeddings_anchor = optimized_model(anchor_embeds)
        ce_loss = criterion_ce(logits_anchor, anchor_labels)

        _, embeddings_paired = optimized_model(paired_embeds)
        contrastive_loss = criterion_contrastive(embeddings_anchor, embeddings_paired, pair_labels)

        # Dynamic Loss Balance
        c1_val = torch.exp(c1)
        c2_val = torch.exp(c2)
        total_loss = c1_val * ce_loss + c2_val * contrastive_loss

        total_loss.backward()
        optimizer_model.step()
        optimizer_weights.step()

        total_loss_epoch += total_loss.item()
        total_ce_loss_epoch += ce_loss.item()
        total_contrastive_loss_epoch += contrastive_loss.item()

    # Calculate total steps for averaging
    num_batches = (num_train_samples + batch_size - 1) // batch_size
    avg_train_loss = total_loss_epoch / num_batches
    avg_train_ce_loss = total_ce_loss_epoch / num_batches
    avg_train_contrastive_loss = total_contrastive_loss_epoch / num_batches

    # validation Every 2 epochs
    if (epoch + 1) % 2 == 0 or epoch == 0 or epoch == (num_epochs - 1):
        optimized_model.eval()
        val_correct_predictions = 0
        val_total_samples = val_embeddings_tensor.size(0)
        val_loss_epoch = 0

        with torch.no_grad():
            # Slice validation data directly on GPU as well
            for i in range(0, val_total_samples, batch_size):
                anchor_embeds = val_embeddings_tensor[i:i + batch_size]
                anchor_labels = val_labels_tensor[i:i + batch_size]

                logits_anchor, _ = optimized_model(anchor_embeds)
                val_loss_epoch += criterion_ce(logits_anchor, anchor_labels).item()

                _, predicted = torch.max(logits_anchor.data, 1)
                val_correct_predictions += (predicted == anchor_labels).sum().item()

        num_val_batches = (val_total_samples + batch_size - 1) // batch_size
        avg_val_loss = val_loss_epoch / num_val_batches
        val_accuracy = val_correct_predictions / val_total_samples

        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} (CE: {avg_train_ce_loss:.4f}, Contrastive: {avg_train_contrastive_loss:.4f}) | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f} | c1: {c1_val.item():.4f} | c2: {c2_val.item():.4f}")

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_wts = copy.deepcopy(optimized_model.state_dict())
            print(f"    >>> New best model saved with Val Acc: {best_val_accuracy:.4f}")
    else:
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} (Skipping Validation)")

optimized_model.load_state_dict(best_model_wts)
print(f"Training complete. Best Validation Accuracy: {best_val_accuracy:.4f}")

Training on cuda...
Epoch 1/20 - Train Loss: 0.6131 (CE: 0.3319, Contrastive: 0.2817) | Val Loss: 0.1957 | Val Acc: 0.9274 | c1: 0.9987 | c2: 0.9981
    >>> New best model saved with Val Acc: 0.9274
Epoch 2/20 - Train Loss: 0.4637 (CE: 0.2036, Contrastive: 0.2612) | Val Loss: 0.1763 | Val Acc: 0.9305 | c1: 0.9974 | c2: 0.9963
    >>> New best model saved with Val Acc: 0.9305
Epoch 3/20 - Train Loss: 0.4338 (Skipping Validation)
Epoch 4/20 - Train Loss: 0.4179 (CE: 0.1726, Contrastive: 0.2477) | Val Loss: 0.1529 | Val Acc: 0.9405 | c1: 0.9947 | c2: 0.9927
    >>> New best model saved with Val Acc: 0.9405
Epoch 5/20 - Train Loss: 0.4051 (Skipping Validation)
Epoch 6/20 - Train Loss: 0.3958 (CE: 0.1564, Contrastive: 0.2430) | Val Loss: 0.1302 | Val Acc: 0.9519 | c1: 0.9919 | c2: 0.9891
    >>> New best model saved with Val Acc: 0.9519
Epoch 7/20 - Train Loss: 0.3895 (Skipping Validation)
Epoch 8/20 - Train Loss: 0.3809 (CE: 0.1446, Contrastive: 0.2411) | Val Loss: 0.1202 | Val Acc: 0.9519

In [19]:
def calc_acc(output_tuple):
    results = output_tuple[0]
    refs = output_tuple[1]
    if results and refs:
        accuracy = accuracy_score(refs, results)
        print(f"/nAccuracy: {accuracy:.4f}")
        print("/nClassification Report:")
        print(classification_report(refs, results))

        print(f"/nPrediction distribution:")
        from collections import Counter
        print("Predictions:", Counter(results))
        print("True labels:", Counter(refs))

In [20]:
def evaluate_model_nn(model, data_df, embeddings, label_map, device, batch_size=2048):
    model.eval()  # Set the model to evaluation mode

    # Invert label_map to convert numeric predictions back to original labels
    inverse_label_map = {v: k for k, v in label_map.items()}

    # 1. Convert ALL embeddings to a single GPU tensor instantly
    # (If 'embeddings' is already a PyTorch tensor on GPU, this line costs nothing)
    if not isinstance(embeddings, torch.Tensor):
        embeddings_tensor = torch.from_numpy(embeddings).float().to(device)
    else:
        embeddings_tensor = embeddings.to(device)

    all_predicted_classes = []
    num_samples = len(data_df)

    # 2. Process in fast vectorized batches
    with torch.no_grad():
        for i in range(0, num_samples, batch_size):
            batch_embeds = embeddings_tensor[i:i + batch_size]

            logits, _ = model(batch_embeds)
            _, predicted_classes = torch.max(logits, 1)

            # Collect the raw numeric predictions
            all_predicted_classes.extend(predicted_classes.cpu().tolist())

    # 3. Map back to string labels using fast list comprehensions
    predictions = [inverse_label_map[num] for num in all_predicted_classes]
    true_labels = data_df['result'].tolist()

    return predictions, true_labels
print("Neural Network evaluation function defined.")

Neural Network evaluation function defined.


In [21]:
# K folds setup

from sklearn.model_selection import StratifiedKFold

# Configuration
n_splits = 5

# combine datasets
df = pd.concat([neutral_df, biased_df], ignore_index=True)
embeddings = sbert_encoder.encode(df['title'].tolist(), show_progress_bar=True, convert_to_numpy=True)
y = df['result'].values  # Target array for stratification
y_numeric = df['result'].map(label_map).values

# Initialize K-Fold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)



Batches:   0%|          | 0/4024 [00:00<?, ?it/s]

In [22]:
def train_fold(model, train_embeds, train_labels, val_embeds, val_labels, device, epochs=20):
    c1 = nn.Parameter(torch.tensor(0.0, device=device, requires_grad=True))
    c2 = nn.Parameter(torch.tensor(-2.0, device=device, requires_grad=True))

    optimizer_model = torch.optim.Adam(model.parameters(), lr=0.0001)
    optimizer_weights = torch.optim.Adam([c1, c2], lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_model, mode='max', factor=0.5, patience=3)

    criterion_ce = nn.CrossEntropyLoss(weight=class_weights).to(device)
    criterion_contrastive = ContrastiveLoss(margin=1.0).to(device)

    # Generate dynamic pairs local to this fold
    pair_indices = torch.zeros(len(train_labels), dtype=torch.long, device=device)
    for label in torch.unique(train_labels):
        mask = (train_labels == label)
        matching_indices = mask.nonzero(as_tuple=True)[0]
        pair_indices[mask] = matching_indices[torch.randint(0, len(matching_indices), (mask.sum(),))]

    best_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    batch_size = 512

    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(train_embeds.size(0), device=device)
        for i in range(0, train_embeds.size(0), batch_size):
            indices = permutation[i:i + batch_size]
            optimizer_model.zero_grad()
            optimizer_weights.zero_grad()

            logits_anchor, embeds_anchor = model(train_embeds[indices])
            ce_loss = criterion_ce(logits_anchor, train_labels[indices])

            _, embeds_paired = model(train_embeds[pair_indices[indices]])
            contrastive_loss = criterion_contrastive(embeds_anchor, embeds_paired, train_labels[indices])

            total_loss = torch.exp(c1) * ce_loss + torch.exp(c2) * contrastive_loss
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer_model.step()
            optimizer_weights.step()

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for i in range(0, val_embeds.size(0), batch_size):
                logits, _ = model(val_embeds[i:i + batch_size])
                val_correct += (logits.argmax(1) == val_labels[i:i + batch_size]).sum().item()

        val_acc = val_correct / val_embeds.size(0)
        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = copy.deepcopy(model.state_dict())
        scheduler.step(val_acc)

    model.load_state_dict(best_wts)
    return model

In [23]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(embeddings, y_numeric)):
    print(f"--- Fold {fold + 1}/5 ---")

    # 1. Fresh model for this fold
    fold_model = BiasedNeutralModel(embedding_dim=embedding_dim, num_classes=2).to(device)

    # 2. Slice only the primary data
    t_embeds = torch.from_numpy(embeddings[train_idx]).float().to(device)
    t_labels = torch.from_numpy(y_numeric[train_idx]).long().to(device)

    v_embeds = torch.from_numpy(embeddings[val_idx]).float().to(device)
    v_labels = torch.from_numpy(y_numeric[val_idx]).long().to(device)

    # 3. Train using the Dynamic Pairing function (no paired args needed)
    fold_model = train_fold(
        model=fold_model,
        train_embeds=t_embeds,
        train_labels=t_labels,
        val_embeds=v_embeds,
        val_labels=v_labels,
        device=device,
        epochs=20
    )

    # 4. Evaluate
    val_df_fold = df.iloc[val_idx]
    preds, truths = evaluate_model_nn(
        model=fold_model,
        data_df=val_df_fold,
        embeddings=embeddings[val_idx],
        label_map=label_map,
        device=device
    )

    print(classification_report(truths, preds))

--- Fold 1/5 ---
              precision    recall  f1-score   support

      biased       0.91      0.92      0.92      9121
     neutral       0.96      0.95      0.95     16629

    accuracy                           0.94     25750
   macro avg       0.93      0.94      0.93     25750
weighted avg       0.94      0.94      0.94     25750

--- Fold 2/5 ---
              precision    recall  f1-score   support

      biased       0.90      0.93      0.92      9121
     neutral       0.96      0.95      0.95     16629

    accuracy                           0.94     25750
   macro avg       0.93      0.94      0.93     25750
weighted avg       0.94      0.94      0.94     25750

--- Fold 3/5 ---
              precision    recall  f1-score   support

      biased       0.91      0.92      0.91      9121
     neutral       0.96      0.95      0.95     16629

    accuracy                           0.94     25750
   macro avg       0.93      0.93      0.93     25750
weighted avg       0.94

In [26]:
import gc
import torch
import warnings
from transformers import pipeline, AutoTokenizer
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

# Clear memory before loading
torch.cuda.empty_cache()
gc.collect()

model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

pipe = pipeline(
    "text-generation", 
    model=model_id, 
    device=0, 
    torch_dtype=torch.float16
)

test_subset = test_df_shuffled.head(100).copy()
predictions = []

print("Running Self-Correcting Two-Step Prompt Pipeline...")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
    for title in tqdm(test_subset['title'].tolist()):
        # --- STEP 1: INITIAL ANALYSIS ---
        messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert media analyst. Analyze the following headlines. First, explain your reasoning briefly, "
                    "then conclude your analysis with exactly: 'Classification: neutral' or 'Classification: biased'.\n"
                    "Rules:\n"
                    "- Neutral headlines rely strictly on factual data and avoid subjective modifiers.\n"
                    "- Biased headlines use emotional adjectives, loaded verbs, or one-sided spin."
                )
            },
            {"role": "user", "content": 'Headline: "Unemployment rates steady as market enters third quarter"'},
            {"role": "assistant", "content": "Reasoning: The headline states matter-of-fact economic indicators without emotional words.\nClassification: neutral"},
            {"role": "user", "content": 'Headline: "Corrupt politicians completely ruin local municipal budget"'},
            {"role": "assistant", "content": "Reasoning: Words like 'corrupt' and 'ruin' add strong negative bias and subjective judgment.\nClassification: biased"},
            {"role": "user", "content": f'Headline: "{title}"'}
        ]
        
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + "Reasoning:"
        
        result = pipe(prompt, max_new_tokens=45, return_full_text=False, temperature=0.1, do_sample=False)
        output_text = result[0]["generated_text"].strip().lower()
        
        # Determine initial classification
        if "classification: neutral" in output_text:
            initial_pred = "neutral"
        elif "classification: biased" in output_text:
            initial_pred = "biased"
        else:
            initial_pred = "neutral" if "neutral" in output_text else "biased"
            
        # If the model guessed biased, force it to cross-examine its own choice
        if initial_pred == "biased":
            verification_messages = [
                {
                    "role": "system",
                    "content": "You are a strict editor. Review the headline classification. Double-check if the headline is truly biased, or if it is just presenting negative facts in a neutral manner. Answer with exactly 'biased' or 'neutral'."
                },
                {"role": "user", "content": f"Headline: \"{title}\"\nIs this headline truly 'biased' or actually 'neutral'?"}
            ]
            
            ver_prompt = tokenizer.apply_chat_template(verification_messages, tokenize=False, add_generation_prompt=True)
            ver_result = pipe(ver_prompt, max_new_tokens=10, return_full_text=False, temperature=0.1, do_sample=False)
            ver_output = ver_result[0]["generated_text"].strip().lower()
            
            if "neutral" in ver_output:
                predictions.append("neutral")
            else:
                predictions.append("biased")
        else:
            predictions.append("neutral")

# Generate and display report metrics
test_subset['pipeline_predicted'] = predictions

print("\n--- Optimized Pipeline Evaluation Results ---")
print(f"Subset Accuracy: {accuracy_score(test_subset['result'], test_subset['pipeline_predicted']):.4f}")
print("\nClassification Report:")
print(classification_report(test_subset['result'], test_subset['pipeline_predicted'], zero_division=0))

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Running Self-Correcting Two-Step Prompt Pipeline...


  1%|          | 1/100 [00:04<08:12,  4.98s/it][transformers] Both `max_new_tokens` (=45) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  3%|▎         | 3/100 [00:19<10:57,  6.78s/it][transformers] Both `max_new_tokens` (=45) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take pr


--- Optimized Pipeline Evaluation Results ---
Subset Accuracy: 0.6600

Classification Report:
              precision    recall  f1-score   support

      biased       0.50      0.76      0.60        34
     neutral       0.83      0.61      0.70        66

    accuracy                           0.66       100
   macro avg       0.67      0.69      0.65       100
weighted avg       0.72      0.66      0.67       100

